# Ingestion Bronze : DVF vers le Lakehouse `Bronze_immo`

Ce notebook télécharge les fichiers DVF géolocalisés (data.gouv.fr), les dépose dans `Files/raw/dvf` du Lakehouse, puis les convertit en table Delta `bronze_dvf_mutations`.

**Prérequis** : rattacher ce notebook au Lakehouse `Bronze_immo` (lakehouse par défaut).

La couche Bronze reste en texte brut : le typage est fait en Silver avec dbt.

## 1. Téléchargement dans `Files/raw/dvf`

In [ ]:
import os, requests

BASE_URL = "https://files.data.gouv.fr/geo-dvf/latest/csv"
YEARS = [2022, 2023, 2024]
DEPTS = ["75", "91", "92", "93", "94"]   # Paris, Essonne, petite couronne
RAW_DIR = "/lakehouse/default/Files/raw/dvf"
os.makedirs(RAW_DIR, exist_ok=True)

for year in YEARS:
    for dept in DEPTS:
        url = f"{BASE_URL}/{year}/departements/{dept}.csv.gz"
        dest = f"{RAW_DIR}/dvf_{year}_{dept}.csv.gz"
        if os.path.exists(dest):
            print(f"déjà présent : {dest}")
            continue
        r = requests.get(url, stream=True, timeout=120)
        if r.status_code != 200:
            print(f"ÉCHEC {r.status_code} : {url}")
            continue
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
        print(f"OK : {dest}")


## 2. Ingestion en table Delta Bronze

In [ ]:
from pyspark.sql import functions as F

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)      # Bronze = tout en texte, le typage se fait en Silver avec dbt
    .csv("Files/raw/dvf/*.csv.gz")
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_dvf_mutations")
)


## 3. Contrôle

In [ ]:
spark.sql("""
    SELECT substr(date_mutation, 1, 4) AS annee,
           code_departement,
           COUNT(*) AS nb_lignes
    FROM bronze_dvf_mutations
    GROUP BY 1, 2
    ORDER BY 1, 2
""").show(30)
